# 00 Extract base data (EU) - urban/rural reference datasets — download & prepare

**Run `00_a_prep_ua_data.ipynb` first** -- the Ukrainian rows of the NUTS and LAU subsets and
the UA population used for GDP per head come from the files it builds in `data/ukraine/`.

**Run order:** `00_a_prep_ua_data` -> this notebook §1-2b -> `00_b1`...`00_b5`
(`*_prep_population_municipality`, one per country) -> this notebook **§2c** (+§3). §2c attaches
the 2025 municipal population the `00_b*` notebooks write; it prints `[missing]` and skips until
all five dumps exist, and must be re-run after any rebuild of §2a/2b.

Companion notebook to the Košice rail-connectivity study. Each section
downloads one open dataset and builds the CZ/SK/AT/HU/PL/UA study-area subset
(`subsetv4atua`) the pipeline reads.

| # | Dataset | What it gives you | Granularity | Coverage | Format |
|---|---|---|---|---|---|
| 1 | **Eurostat NUTS 2024** | Country (NUTS-0) + region (NUTS-1/2/3) polygons | NUTS-0/1/2/3 | EU + EFTA + candidates (UA level 0 only) | GeoJSON |
| 2a | **Eurostat LAU 2024** | Municipality polygons + population, area | Municipality (LAU) | EU + EFTA; UA hromadas appended | GeoJSON |
| 2b | **Eurostat DEGURBA 2021** | Official 3-category city/town/rural label per municipality | Municipality (LAU) | EU + EFTA; UA from GHSL grid | Shapefile |
| 2c | **National statistics, end-2025** (via `00_b1`...`00_b5`) | Population total + 0-14 / 15-64 / 65+ per municipality | Municipality (LAU) | AT, CZ, HU, PL, SK | CSV -> GeoJSON |
| 2d | **LAU 2025 workbook** (`EU-27-LAU-2025-NUTS-2024.xlsx`) + UA hromadas | LAU / EU LAU codes, national and Latin names, NUTS3 / NUTS2 per LAU | Municipality (LAU) | AT, CZ, HU, PL, SK; UA from HDX | XLSX -> GeoJSON |

**Important caveats**

- **URLs change.** Eurostat and JRC reorganise download paths periodically.
  Where I'm not 100% sure the URL is current I leave a `# VERIFY` comment —
  open the dataset's landing page and grab the latest link if the download
  fails.
- **Coverage gaps for Ukraine.** NUTS, LAU, and DEGURBA stop at EU/EFTA
  borders. The study-area subsets append Ukrainian oblasts and hromadas from
  `data/ukraine/` (built by `input_data_extension_ukraine.ipynb`).
- **Licensing.** All are open. Most are CC-BY or equivalent; cite as noted in
  each section.
Run the sections in order: 2b attaches DEGURBA to the LAU subset built in 2a.

## Setup

Project-root working directory, `cfg`, and the `download()` / `unzip()` helpers.

In [1]:
import sys, os

# Ensure CWD is the project root (one level up from notebooks/)
_here = os.path.abspath('')
if os.path.basename(_here) == 'notebooks':
    os.chdir(os.path.dirname(_here))
sys.path.insert(0, os.getcwd())
print('Working directory:', os.getcwd())

from traintran.config import default_config
cfg = default_config()

Working directory: E:\notes\api_zssk\project


In [2]:
from pathlib import Path
import os, sys, zipfile, shutil
import requests

# Polite default headers so servers don't mistake us for a scraper.
HEADERS = {"User-Agent": "academic-research/1.0 (KSC rail study)"}

def download(url, dest, force=False):
    """Stream-download `url` to `dest`. Skip if cached unless force=True."""
    dest = Path(dest)
    if dest.exists() and not force and dest.stat().st_size > 0:
        print(f"  [cached] {dest.name} ({dest.stat().st_size/1e6:.1f} MB)")
        return dest
    print(f"  [fetch]  {url}")
    with requests.get(url, headers=HEADERS, stream=True, timeout=180) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                f.write(chunk)
    print(f"  [saved]  {dest} ({dest.stat().st_size/1e6:.1f} MB)")
    return dest

def unzip(zip_path, out_dir):
    out_dir = Path(out_dir); out_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(out_dir)
    print(f"  [unzipped] -> {out_dir}")
    return out_dir

## 1. Eurostat NUTS 2024 - country & region polygons

Replaces the NUTS 2021 PostGIS table `public.nuts_rg_01m_2021_4326`; the pipeline now reads
these files directly (`traintran.enrichment.load_nuts_level`, `cfg.nuts_dir`).

- **Download:** <https://gisco-services.ec.europa.eu/distribution/v2/nuts/download/ref-nuts-2024-01m.geojson.zip>
- **Landing page:** <https://ec.europa.eu/eurostat/web/gisco/geodata/statistical-units/territorial-units-statistics>
- **License:** Free reuse with attribution to Eurostat / (c) EuroGeographics.
- `RG` = region polygons (`BN` = boundary lines, `LB` = label points are not kept); `4326` = EPSG code.
- GISCO's NUTS 2024 has **Ukraine only at level 0** (see the release notes). The 27 oblasts from
  `data/ukraine/admin_boundaries/ukr_admin1.shp` are appended to the subset as level 2, as
  `input_data_extension_ukraine.ipynb` previously did in PostGIS; `SOURCE` marks those rows.
- Note: the `metadata.pdf` GISCO ships in this package still describes NUTS 2021.

In [3]:
import zipfile
from pathlib import Path

NUTS_URL = ("https://gisco-services.ec.europa.eu/distribution/v2/nuts/"
            "download/ref-nuts-2024-01m.geojson.zip")
NUTS_DIR = Path("data/eu/ref-nuts-2024-01m")
NUTS_KEEP = [
    "NUTS_RG_01M_2024_4326.geojson",
    *[f"NUTS_RG_01M_2024_4326_LEVL_{n}.geojson" for n in range(4)],
    "metadata.pdf",
]

NUTS_DIR.mkdir(parents=True, exist_ok=True)
present = [f for f in NUTS_KEEP if (NUTS_DIR / f).is_file() and (NUTS_DIR / f).stat().st_size > 0]

if len(present) == len(NUTS_KEEP):
    for f in NUTS_KEEP:
        print(f"  [exists]    {NUTS_DIR / f}")
    print("  All NUTS 2024 files already in place -- skipping download and extract.")
else:
    nuts_zip = download(NUTS_URL, Path("data/eu") / "ref-nuts-2024-01m.geojson.zip")

    # Extract only the files we need; members may sit in a subfolder, so match on basename.
    with zipfile.ZipFile(nuts_zip) as z:
        by_name = {Path(m).name: m for m in z.namelist() if not m.endswith("/")}
        missing = [f for f in NUTS_KEEP if f not in by_name]
        if missing:
            raise FileNotFoundError(f"Not in {nuts_zip.name}: {missing}")
        for f in NUTS_KEEP:
            if f in present:
                print(f"  [exists]    {NUTS_DIR / f}")
                continue
            with z.open(by_name[f]) as src, open(NUTS_DIR / f, "wb") as dst:
                shutil.copyfileobj(src, dst)
            print(f"  [extracted] {NUTS_DIR / f}")

    nuts_zip.unlink()
    print(f"  [deleted]   {nuts_zip}")


  [exists]    data\eu\ref-nuts-2024-01m\NUTS_RG_01M_2024_4326.geojson
  [exists]    data\eu\ref-nuts-2024-01m\NUTS_RG_01M_2024_4326_LEVL_0.geojson
  [exists]    data\eu\ref-nuts-2024-01m\NUTS_RG_01M_2024_4326_LEVL_1.geojson
  [exists]    data\eu\ref-nuts-2024-01m\NUTS_RG_01M_2024_4326_LEVL_2.geojson
  [exists]    data\eu\ref-nuts-2024-01m\NUTS_RG_01M_2024_4326_LEVL_3.geojson
  [exists]    data\eu\ref-nuts-2024-01m\metadata.pdf
  All NUTS 2024 files already in place -- skipping download and extract.


### 1a. Study-area subset `subsetv4atua` (CZ, SK, AT, HU, PL, UA)

Same filenames as the source, one per level plus the all-levels file. UA has a GISCO row
at level 0 only; its level 2 comes from `ukr_admin1`.

In [4]:
import geopandas as gpd
import pandas as pd

SUBSET_COUNTRIES = ["CZ", "SK", "AT", "HU", "PL", "UA"]
SUBSET_DIR = NUTS_DIR / "subsetv4atua"
SUBSET_DIR.mkdir(parents=True, exist_ok=True)

# UA oblasts in the GISCO schema -- same mapping as input_data_extension_ukraine.ipynb.
ukr_admin1 = gpd.read_file("data/ukraine/admin_boundaries/ukr_admin1.shp").to_crs(4326)
ua_level2 = gpd.GeoDataFrame({
    "NUTS_ID":   ukr_admin1["adm1_pcode"],
    "LEVL_CODE": 2,
    "CNTR_CODE": ukr_admin1["adm0_pcode"],   # 'UA'
    "NAME_LATN": ukr_admin1["adm1_name"],
    "NUTS_NAME": ukr_admin1["adm1_name1"],
    "SOURCE":    "HDX_ukr_admin1",
}, geometry=ukr_admin1.geometry, crs=4326)

counts = []
for fname in NUTS_KEEP[:-1]:                       # every .geojson, not metadata.pdf
    gdf = gpd.read_file(NUTS_DIR / fname)
    gdf = gdf[gdf["CNTR_CODE"].isin(SUBSET_COUNTRIES)].copy()
    gdf["SOURCE"] = "GISCO_NUTS_2024"
    if fname.endswith("_4326.geojson") or fname.endswith("_LEVL_2.geojson"):
        gdf = gpd.GeoDataFrame(pd.concat([gdf, ua_level2], ignore_index=True), crs=4326)
    gdf.to_file(SUBSET_DIR / fname, driver="GeoJSON")
    counts.append(gdf.assign(file=fname).groupby(["file", "LEVL_CODE", "CNTR_CODE"]).size())
    print(f"  [subset] {SUBSET_DIR / fname}  ({len(gdf):,} rows)")

pd.concat(counts).unstack("CNTR_CODE", fill_value=0)

  [subset] data\eu\ref-nuts-2024-01m\subsetv4atua\NUTS_RG_01M_2024_4326.geojson  (244 rows)


  [subset] data\eu\ref-nuts-2024-01m\subsetv4atua\NUTS_RG_01M_2024_4326_LEVL_0.geojson  (6 rows)


  [subset] data\eu\ref-nuts-2024-01m\subsetv4atua\NUTS_RG_01M_2024_4326_LEVL_1.geojson  (15 rows)


  [subset] data\eu\ref-nuts-2024-01m\subsetv4atua\NUTS_RG_01M_2024_4326_LEVL_2.geojson  (73 rows)


  [subset] data\eu\ref-nuts-2024-01m\subsetv4atua\NUTS_RG_01M_2024_4326_LEVL_3.geojson  (150 rows)


CNTR_CODE                                       AT  CZ  HU  PL  SK  UA
file                                 LEVL_CODE                        
NUTS_RG_01M_2024_4326.geojson        0           1   1   1   1   1   1
                                     1           3   1   3   7   1   0
                                     2           9   8   8  17   4  27
                                     3          35  14  20  73   8   0
NUTS_RG_01M_2024_4326_LEVL_0.geojson 0           1   1   1   1   1   1
NUTS_RG_01M_2024_4326_LEVL_1.geojson 1           3   1   3   7   1   0
NUTS_RG_01M_2024_4326_LEVL_2.geojson 2           9   8   8  17   4  27
NUTS_RG_01M_2024_4326_LEVL_3.geojson 3          35  14  20  73   8   0

In [5]:
# Every catchment NUTS prefix must exist in the 2024 subset.
from traintran.eu_lau_catchment import CATCHMENTS
from traintran.enrichment import load_nuts_level

_ids = set(pd.concat([load_nuts_level(cfg, n)["nuts_id"] for n in range(4)]))
for name, spec in CATCHMENTS.items():
    bad = [p for p in spec["nuts_prefixes"] if not any(i.startswith(p) for i in _ids)]
    print(f"  {name:<16} {'OK' if not bad else 'MISSING ' + str(bad)}")

  canonical        OK
  canonical_ua     OK
  sk_only          OK
  sk_all           OK
  sk_all_incl_ba   OK
  hu31_ua          OK


## 2a. Eurostat LAU 2024 - municipality polygons

Replaces the LAU 2021 shapefile. One polygon per Local Administrative Unit (municipality),
with population, density and area.

- **Download:** <https://gisco-services.ec.europa.eu/distribution/v2/lau/download/ref-lau-2024-01m.geojson.zip>
- **Landing page:** <https://ec.europa.eu/eurostat/web/gisco/geodata/local-administrative-units>
- **License:** Free reuse with attribution to Eurostat.
- **Fields:** `GISCO_ID` (= `EU LAU CODE` in `EU-27-LAU-2025-NUTS-2024.xlsx`), `CNTR_CODE`,
  `LAU_NAME`, `POP_2024`, `POP_DENS_2024`, `AREA_KM2`, `YEAR`.
- **Coverage gap:** EU + EFTA only - no Ukrainian LAUs. The subset appends the 1,769 hromadas
  (Ukraine's LAU equivalent) from `data/ukraine/ukr_admin3_population.gpkg`, built by
  `input_data_extension_ukraine.ipynb`. Their `GISCO_ID` is the `adm3_pcode`, and their
  population is the State Statistics estimate for **1 Jan 2022** -- see `POP_YEAR`; `SOURCE`
  marks the non-Eurostat rows.

In [6]:
LAU_URL = ("https://gisco-services.ec.europa.eu/distribution/v2/lau/"
           "download/ref-lau-2024-01m.geojson.zip")
LAU_DIR = Path("data/eu/ref-lau-2024-01m")
LAU_FILE = "LAU_RG_01M_2024_4326.geojson"
LAU_OPTIONAL = ["metadata.pdf", "metadata.xml", "release-notes.txt"]   # copied if the zip has them

LAU_DIR.mkdir(parents=True, exist_ok=True)
if (LAU_DIR / LAU_FILE).is_file() and (LAU_DIR / LAU_FILE).stat().st_size > 0:
    print(f"  [exists]    {LAU_DIR / LAU_FILE}")
    print("  LAU 2024 already in place -- skipping download and extract.")
else:
    lau_zip = download(LAU_URL, Path("data/eu") / "ref-lau-2024-01m.geojson.zip")

    # Extract only the files we need; members may sit in a subfolder, so match on basename.
    with zipfile.ZipFile(lau_zip) as z:
        by_name = {Path(m).name: m for m in z.namelist() if not m.endswith("/")}
        if LAU_FILE not in by_name:
            raise FileNotFoundError(f"{LAU_FILE} not in {lau_zip.name}; it has: {sorted(by_name)}")
        for f in [LAU_FILE] + [o for o in LAU_OPTIONAL if o in by_name]:
            with z.open(by_name[f]) as src, open(LAU_DIR / f, "wb") as dst:
                shutil.copyfileobj(src, dst)
            print(f"  [extracted] {LAU_DIR / f}")
        not_found = [o for o in LAU_OPTIONAL if o not in by_name]
        if not_found:
            print(f"  (not in zip: {not_found})")

    lau_zip.unlink()
    print(f"  [deleted]   {lau_zip}")

  [exists]    data\eu\ref-lau-2024-01m\LAU_RG_01M_2024_4326.geojson
  LAU 2024 already in place -- skipping download and extract.


In [7]:
import geopandas as gpd
import pandas as pd

REBUILD_LAU_SUBSET = False          # True -> rebuild even if the subset file already exists
LAU_SUBSET_COUNTRIES = ["CZ", "SK", "AT", "HU", "PL"]   # UA appended from the hromadas below
LAU_SUBSET_DIR = LAU_DIR / "subsetv4atua"
LAU_SUBSET_FILE = LAU_SUBSET_DIR / LAU_FILE
UA_HROMADAS = Path("data/ukraine/ukr_admin3_population.gpkg")

if LAU_SUBSET_FILE.is_file() and LAU_SUBSET_FILE.stat().st_size > 0 and not REBUILD_LAU_SUBSET:
    print(f"  [exists]    {LAU_SUBSET_FILE} -- skipping (set REBUILD_LAU_SUBSET = True to rebuild)")
    lau_subset = gpd.read_file(LAU_SUBSET_FILE)
else:
    if not UA_HROMADAS.is_file():
        raise FileNotFoundError(f"{UA_HROMADAS} missing -- build it with input_data_extension_ukraine.ipynb")
    LAU_SUBSET_DIR.mkdir(parents=True, exist_ok=True)

    # The source covers ~98k LAUs (150 MB); `where` filters while reading.
    _cc = ", ".join(f"'{c}'" for c in LAU_SUBSET_COUNTRIES)
    lau_eu = gpd.read_file(LAU_DIR / LAU_FILE, where=f"CNTR_CODE IN ({_cc})").to_crs(4326)
    lau_eu["POP_YEAR"] = 2024
    lau_eu["SOURCE"] = "GISCO_LAU_2024"

    # UA hromadas in the GISCO LAU schema.
    hrom = gpd.read_file(UA_HROMADAS, layer="hromadas").to_crs(4326)
    lau_ua = gpd.GeoDataFrame({
        "GISCO_ID":      hrom["adm3_pcode"],
        "CNTR_CODE":     "UA",
        "LAU_NAME":      hrom["adm3_name"],
        "POP_2024":      hrom["population"],
        "POP_DENS_2024": hrom["population"] / hrom["area_sqkm"],
        "AREA_KM2":      hrom["area_sqkm"],
        "YEAR":          2024,
        "POP_YEAR":      2022,
        "SOURCE":        "HDX_ukr_admin3+UkrStat_2022",
    }, geometry=hrom.geometry, crs=4326)

    lau_subset = gpd.GeoDataFrame(pd.concat([lau_eu, lau_ua], ignore_index=True), crs=4326)
    assert lau_subset["GISCO_ID"].is_unique, "duplicate GISCO_ID in LAU subset"
    lau_subset.to_file(LAU_SUBSET_FILE, driver="GeoJSON")
    print(f"  [subset]    {LAU_SUBSET_FILE}  ({len(lau_subset):,} rows)")
    print(f"  UA hromadas without population: {int(lau_ua['POP_2024'].isna().sum())} "
          "(occupied Crimea/Sevastopol expected)")

lau_subset.groupby(["CNTR_CODE", "SOURCE"]).size().rename("rows").to_frame()

  [exists]    data\eu\ref-lau-2024-01m\subsetv4atua\LAU_RG_01M_2024_4326.geojson -- skipping (set REBUILD_LAU_SUBSET = True to rebuild)


,,rows
CNTR_CODE,SOURCE,
AT,GISCO_LAU_2024,2092
CZ,GISCO_LAU_2024,6258
HU,GISCO_LAU_2024,3155
PL,GISCO_LAU_2024,2477
SK,GISCO_LAU_2024,2927
UA,HDX_ukr_admin3+UkrStat_2022,1769


## 2b. DEGURBA 2021 - urban/town/rural per LAU

**This is the dataset to use for an official per-station urban/rural label.** GISCO's DEGURBA
polygons carry the classification directly in `DGURBA`: **1 = city, 2 = town/suburb, 3 = rural**.

- **Download:** <https://ec.europa.eu/eurostat/documents/7116161/15125332/DGURBA_RG_01M_2021_4258.zip>
- **DEGURBA landing page:** <https://ec.europa.eu/eurostat/web/degree-of-urbanisation>
- **License:** Free reuse with attribution to Eurostat.
- **Fields:** `GISCO_ID, CNTR_CODE, LAU_ID, LAU_NAME, DGURBA`; EPSG:4258. Extracted to
  `data/eu/ref-dgurba-2021-01m/` (`cfg.dgurba_shp`, used by station tagging).
- **Vintage:** the polygons are **LAU 2021**, but `GISCO_ID` joins 16,909 of the 16,910 LAU 2024
  rows of the subset, with values identical to `DEGURBA` in `EU-27-LAU-2025-NUTS-2024.xlsx`; the
  one AT LAU missing from 2021 is filled from that workbook.
- **Ukraine** (not in Eurostat DEGURBA): `degurba_l1_ghsl` from
  `data/ukraine/ukr_admin3_population.gpkg`, the rule `input_data_extension_ukraine.ipynb` uses
  (GHSL `UKR_DUG_2024` grid; urban centre by containment, town if >= 15 % non-rural area). GHSL
  codes 1 = rural ... 3 = urban centre, so it is flipped with `4 - x` onto the EU scale.
  `DGURBA_SOURCE` records which source each row used.
- **Methodological note for the paper:** DEGURBA is defined on a 1 km grid
  first (using GHSL population) and then aggregated to LAUs.
  For peripheral or large LAUs the aggregation can give counter-intuitive
  results (a rail station in a small village inside a large administrative
  city LAU still tags as "city"). Worth noting as a known limitation.

In [8]:
DGURBA_URL = ("https://ec.europa.eu/eurostat/documents/7116161/15125332/"
              "DGURBA_RG_01M_2021_4258.zip/09a07de6-f16e-af72-1178-123259fadefd?t=1748954595744")
DGURBA_DIR = Path("data/eu/ref-dgurba-2021-01m")
_stem = "DGURBA_RG_01M_2021_4258"
DGURBA_REQUIRED = [f"{_stem}{ext}" for ext in (".shp", ".shx", ".dbf", ".prj", ".cpg")]
DGURBA_OPTIONAL = [f"{_stem}{ext}" for ext in (".sbn", ".sbx", ".shp.xml")]

DGURBA_DIR.mkdir(parents=True, exist_ok=True)
present = [f for f in DGURBA_REQUIRED if (DGURBA_DIR / f).is_file() and (DGURBA_DIR / f).stat().st_size > 0]

if len(present) == len(DGURBA_REQUIRED):
    for f in DGURBA_REQUIRED:
        print(f"  [exists]    {DGURBA_DIR / f}")
    print("  DGURBA shapefile already in place -- skipping download and extract.")
else:
    dgurba_zip = download(DGURBA_URL, Path("data/eu") / f"{_stem}.zip")

    # Extract the shapefile parts; members may sit in a subfolder, so match on basename.
    with zipfile.ZipFile(dgurba_zip) as z:
        by_name = {Path(m).name: m for m in z.namelist() if not m.endswith("/")}
        missing = [f for f in DGURBA_REQUIRED if f not in by_name]
        if missing:
            raise FileNotFoundError(f"Not in {dgurba_zip.name}: {missing}")
        for f in DGURBA_REQUIRED + [o for o in DGURBA_OPTIONAL if o in by_name]:
            with z.open(by_name[f]) as src, open(DGURBA_DIR / f, "wb") as dst:
                shutil.copyfileobj(src, dst)
            print(f"  [extracted] {DGURBA_DIR / f}")

    dgurba_zip.unlink()
    print(f"  [deleted]   {dgurba_zip}")

  [exists]    data\eu\ref-dgurba-2021-01m\DGURBA_RG_01M_2021_4258.shp
  [exists]    data\eu\ref-dgurba-2021-01m\DGURBA_RG_01M_2021_4258.shx
  [exists]    data\eu\ref-dgurba-2021-01m\DGURBA_RG_01M_2021_4258.dbf
  [exists]    data\eu\ref-dgurba-2021-01m\DGURBA_RG_01M_2021_4258.prj
  [exists]    data\eu\ref-dgurba-2021-01m\DGURBA_RG_01M_2021_4258.cpg
  DGURBA shapefile already in place -- skipping download and extract.


In [9]:
# Attach DGURBA to the LAU 2024 subset (section 2a), in place.
from traintran.eu_lau_catchment import load_eu_lau

REBUILD_LAU_DGURBA = False          # True -> recompute even if the subset already has DGURBA

lau_subset = gpd.read_file(LAU_SUBSET_FILE)
if "DGURBA" in lau_subset.columns and not REBUILD_LAU_DGURBA:
    print(f"  [exists]    DGURBA already in {LAU_SUBSET_FILE} -- skipping "
          "(set REBUILD_LAU_DGURBA = True to recompute)")
else:
    lau_subset = lau_subset.drop(columns=["DGURBA", "DGURBA_SOURCE"], errors="ignore")
    is_ua = lau_subset["CNTR_CODE"] == "UA"

    # EU: GISCO DEGURBA 2021, joined on GISCO_ID (attributes only).
    _cc = ", ".join(f"'{c}'" for c in LAU_SUBSET_COUNTRIES)
    dg = gpd.read_file(DGURBA_DIR / DGURBA_REQUIRED[0], where=f"CNTR_CODE IN ({_cc})",
                       ignore_geometry=True)
    dgurba = pd.Series(pd.NA, index=lau_subset.index, dtype="Int64")
    source = pd.Series(pd.NA, index=lau_subset.index, dtype="object")
    _gisco = lau_subset["GISCO_ID"].map(dg.drop_duplicates("GISCO_ID").set_index("GISCO_ID")["DGURBA"])
    dgurba[~is_ua] = pd.to_numeric(_gisco[~is_ua], errors="coerce").astype("Int64")
    source[~is_ua & dgurba.notna()] = "GISCO_DGURBA_2021"

    # EU gaps (LAUs new since 2021): DEGURBA column of the LAU 2025 workbook.
    _gap = ~is_ua & dgurba.isna()
    if _gap.any():
        wb = load_eu_lau(cfg.lau_file, countries=LAU_SUBSET_COUNTRIES)
        _wb = lau_subset.loc[_gap, "GISCO_ID"].map(wb.set_index("euc_lau_code")["degurba"])
        dgurba[_gap] = _wb.astype("Int64")
        source[_gap & dgurba.notna()] = "EU_LAU_2025_workbook"
        print(f"  {int(_gap.sum())} EU LAU(s) not in DGURBA 2021, filled from the workbook: "
              f"{lau_subset.loc[_gap, ['GISCO_ID', 'LAU_NAME']].values.tolist()}")

    # UA: GHSL area rule from input_data_extension_ukraine (1 rural..3 urban centre) -> EU 1 city..3 rural.
    hrom = gpd.read_file(UA_HROMADAS, layer="hromadas", ignore_geometry=True)
    _ghsl = lau_subset.loc[is_ua, "GISCO_ID"].map(hrom.set_index("adm3_pcode")["degurba_l1_ghsl"])
    dgurba[is_ua] = (4 - _ghsl).astype("Int64")
    source[is_ua & dgurba.notna()] = "GHSL_UKR_DUG_2024_area_rule"

    lau_subset["DGURBA"] = dgurba
    lau_subset["DGURBA_SOURCE"] = source
    assert lau_subset["DGURBA"].notna().all(),         f"{int(lau_subset['DGURBA'].isna().sum())} LAU(s) without DGURBA"
    assert set(lau_subset["DGURBA"].unique()) <= {1, 2, 3}
    lau_subset.to_file(LAU_SUBSET_FILE, driver="GeoJSON")
    print(f"  [updated]   {LAU_SUBSET_FILE}  (DGURBA, DGURBA_SOURCE)")
    print(lau_subset["DGURBA_SOURCE"].value_counts().to_string())

pd.crosstab(lau_subset["CNTR_CODE"], lau_subset["DGURBA"].rename("DGURBA (1 city, 2 town, 3 rural)"))

  [exists]    DGURBA already in data\eu\ref-lau-2024-01m\subsetv4atua\LAU_RG_01M_2024_4326.geojson -- skipping (set REBUILD_LAU_DGURBA = True to recompute)


"DGURBA (1 city, 2 town, 3 rural)",1,2,3
CNTR_CODE,,,
AT,18,449,1625
CZ,17,647,5594
HU,18,337,2800
PL,67,538,1872
SK,45,314,2568
UA,74,114,1581


## 2c. Population 2025 by age group (national statistical offices) on the LAU subset

Attaches the end-2025 municipal population written by `00_b1`...`00_b5`
(`data/eu/ref-national-statistical-offices/<cc>/lau_population_2025_<cc>.csv`, one common schema)
to the LAU subset of §2a/2b, **next to** GISCO's `POP_2024`:

| column | content |
|---|---|
| `POP_2025` | total population |
| `POP_0_14_2025`, `POP_15_64_2025`, `POP_65PLUS_2025` | exact age-band counts |
| `POP_2025_REF_DATE` | `2025-12-31` (CZ, HU, PL, SK), `2026-01-01` (AT) |
| `POP_2025_SOURCE` | `OGD_bevstandjbab2002_BevStand_2026`, `CZSO_OBY02ET01`, `GUS_tab12_2025-12-31`, `SUSR_om7006rr`, `KSH_TA2025_W` |
| `NSO_LAU_ID` | the national statistical office's own municipality key: AT GKZ, CZ kód obce, HU KSH T-STAR code, PL TERYT (7 digits), SK `soIdLau2` |
| `TERRITORIAL_NOTE` | set on dissolved features, e.g. `merged: AT_62252;AT_62267` |

**Merged municipalities are dissolved.** A dump row with `replaces_lau` (Austria: Söchau 62252 +
Fürstenfeld 62267 -> Fürstenfeld **62280**, territorial status 2026) replaces those subset features
with one feature: geometry dissolved, `POP_2024` and `AREA_KM2` summed, `POP_DENS_2024`
recomputed, other attributes (incl. `DGURBA`) from the most populous part. Already-dissolved
features are left alone, so the cell is idempotent.

UA hromadas get no 2025 values (no age-banded source). Skips if the columns exist
(`REBUILD_LAU_POP = True` to recompute); prints `[missing]` and skips if any dump is absent.

In [10]:
REBUILD_LAU_POP = False            # True -> recompute even if the subset already has POP_2025
POP_DUMP_DIR = Path("data/eu/ref-national-statistical-offices")
POP_COUNTRIES = ["AT", "CZ", "HU", "PL", "SK"]
POP_DUMPS = {c: POP_DUMP_DIR / c.lower() / f"lau_population_2025_{c.lower()}.csv" for c in POP_COUNTRIES}
POP_ADDED = {"pop_total": "POP_2025", "y_0_14": "POP_0_14_2025", "y_15_64": "POP_15_64_2025",
             "y_65plus": "POP_65PLUS_2025", "ref_date": "POP_2025_REF_DATE", "source": "POP_2025_SOURCE",
             "native_id": "NSO_LAU_ID"}

lau_subset = gpd.read_file(LAU_SUBSET_FILE)
_missing = [p for p in POP_DUMPS.values() if not p.is_file()]
if _missing:
    for p in _missing:
        print(f"  [missing]   {p}")
    print("  run 00_b1...00_b5 (*_prep_population_municipality), then re-run this cell -- skipping")
elif set(POP_ADDED.values()) <= set(lau_subset.columns) and not REBUILD_LAU_POP:
    print(f"  [exists]    POP_2025 already in {LAU_SUBSET_FILE} -- skipping "
          "(set REBUILD_LAU_POP = True to recompute)")
else:
    _str = {c: str for c in ["lau_code", "gisco_id", "native_id", "replaces_lau", "includes_native"]}
    pop25 = pd.concat([pd.read_csv(p, dtype=_str) for p in POP_DUMPS.values()], ignore_index=True)
    assert pop25["gisco_id"].is_unique, "duplicate gisco_id across the dumps"

    lau_subset = lau_subset.drop(columns=list(POP_ADDED.values()), errors="ignore")
    if "TERRITORIAL_NOTE" not in lau_subset.columns:
        lau_subset["TERRITORIAL_NOTE"] = None

    # Dissolve merged municipalities into their new code (data-driven: replaces_lau).
    for _, r in pop25[pop25["replaces_lau"].notna()].iterrows():
        olds = [f"{r['country']}_{c}" for c in r["replaces_lau"].split(";")]
        if r["gisco_id"] in set(lau_subset["GISCO_ID"]):
            print(f"  [exists]    {r['gisco_id']} already dissolved")
            continue
        parts = lau_subset[lau_subset["GISCO_ID"].isin(olds)]
        assert len(parts) == len(olds), f"{r['gisco_id']}: subset has {sorted(parts.GISCO_ID)}, expected {olds}"
        new = parts.sort_values("POP_2024", ascending=False).iloc[[0]].copy()
        new["GISCO_ID"] = r["gisco_id"]
        new["LAU_NAME"] = r["lau_name"]
        new["POP_2024"] = parts["POP_2024"].sum()
        new["AREA_KM2"] = parts["AREA_KM2"].sum()
        new["POP_DENS_2024"] = new["POP_2024"] / new["AREA_KM2"]
        new["TERRITORIAL_NOTE"] = "merged: " + ";".join(olds)
        new = new.set_geometry([parts.geometry.union_all()], crs=lau_subset.crs)
        lau_subset = gpd.GeoDataFrame(pd.concat([lau_subset[~lau_subset["GISCO_ID"].isin(olds)], new],
                                                ignore_index=True), crs=lau_subset.crs)
        print(f"  [dissolved] {' + '.join(olds)} -> {r['gisco_id']} ({r['lau_name']}), "
              f"DGURBA {int(new['DGURBA'].iloc[0])}")

    # Attach the 2025 columns on GISCO_ID.
    _by = pop25.set_index("gisco_id")
    for src_col, dst_col in POP_ADDED.items():
        lau_subset[dst_col] = lau_subset["GISCO_ID"].map(_by[src_col])
    for c in ["POP_2025", "POP_0_14_2025", "POP_15_64_2025", "POP_65PLUS_2025"]:
        lau_subset[c] = lau_subset[c].astype("Int64")

    is_ua = lau_subset["CNTR_CODE"] == "UA"
    assert lau_subset["GISCO_ID"].is_unique, "duplicate GISCO_ID after dissolving"
    assert lau_subset.loc[~is_ua, "POP_2025"].notna().all(), \
        f"{int(lau_subset.loc[~is_ua, 'POP_2025'].isna().sum())} EU feature(s) without POP_2025"
    assert set(pop25["gisco_id"]) <= set(lau_subset["GISCO_ID"]), "dump rows without a subset feature"
    _sum = lau_subset[~is_ua].groupby("CNTR_CODE")["POP_2025"].sum()
    assert _sum.equals(pop25.groupby("country")["pop_total"].sum().reindex(_sum.index).astype("Int64")), "totals differ"
    _bands = lau_subset.loc[~is_ua, ["POP_0_14_2025", "POP_15_64_2025", "POP_65PLUS_2025"]].sum(axis=1)
    assert (_bands == lau_subset.loc[~is_ua, "POP_2025"]).all(), "bands do not add up"

    lau_subset.to_file(LAU_SUBSET_FILE, driver="GeoJSON")
    print(f"  [updated]   {LAU_SUBSET_FILE}  ({len(lau_subset):,} features; "
          f"{', '.join(POP_ADDED.values())}, TERRITORIAL_NOTE)")

if "POP_2025" in lau_subset.columns:
    _eu = lau_subset[lau_subset["CNTR_CODE"] != "UA"]
    _t = _eu.groupby("CNTR_CODE").agg(features=("GISCO_ID", "size"), POP_2024=("POP_2024", "sum"),
                                      POP_2025=("POP_2025", "sum"), ref_date=("POP_2025_REF_DATE", "first"))
    for c, b in [("0-14 %", "POP_0_14_2025"), ("15-64 %", "POP_15_64_2025"), ("65+ %", "POP_65PLUS_2025")]:
        _t[c] = (100 * _eu.groupby("CNTR_CODE")[b].sum() / _t["POP_2025"]).astype(float).round(1)
    display(_t)

  [exists]    POP_2025 already in data\eu\ref-lau-2024-01m\subsetv4atua\LAU_RG_01M_2024_4326.geojson -- skipping (set REBUILD_LAU_POP = True to recompute)


,features,POP_2024,POP_2025,ref_date,0-14 %,15-64 %,65+ %
CNTR_CODE,,,,,,,
AT,2092,9158750.0,9215956.0,2026-01-01,14.2,65.2,20.7
CZ,6258,10900555.0,10915839.0,2025-12-31,15.1,64.1,20.8
HU,3155,9584627.0,9488428.0,2025-12-31,14.2,64.7,21.1
PL,2477,37636508.0,37332510.0,2025-12-31,14.2,64.7,21.1
SK,2927,5424687.0,5409407.0,2025-12-31,15.6,65.2,19.3


## 2d. LAU identifiers, names and NUTS codes on the LAU subset

So the subset can be referenced on its own (station tagging no longer needs the workbook or the
DEGURBA shapefile), every feature gets its identifiers and its NUTS codes:

| column | EU features | `AT_62280` (merged, §2c) | UA hromadas |
|---|---|---|---|
| `LAU_CODE` | workbook `LAU CODE` | `62280` | `adm3_pcode` (= `GISCO_ID`) |
| `EU_LAU_CODE` | workbook `EU LAU CODE` (= `GISCO_ID`) | `AT_62280` | empty |
| `LAU_NAME_NATIONAL` | workbook `LAU NAME NATIONAL` | Fürstenfeld | `adm3_name1` (Ukrainian) |
| `LAU_NAME_LATIN` | workbook `LAU NAME LATIN` | Fürstenfeld | `adm3_name` (romanised) |
| `NUTS3_ID` | workbook `NUTS3` | its parts' NUTS3 | empty (GISCO has no UA level 3) |
| `NUTS2_ID` | `NUTS3_ID[:4]` | same | `adm1_pcode` (`UA01` …, the subset's UA level-2 ids) |
| `LAU_ID_SOURCE` | `LAU_2025_NUTS_2024_xlsx` | `merged_2026: AT_62252;AT_62267` | `HDX_ukr_admin3` |

- **Source:** `data/eu/ref-nuts-2024-01m/EU-27-LAU-2025-NUTS-2024.xlsx` (Eurostat LAU 2025 / NUTS 2024,
  byte-identical to `cfg.lau_file`), read with `traintran.eu_lau_catchment.load_eu_lau`; UA names and
  oblast codes from `data/ukraine/ukr_admin3_population.gpkg` (layer `hromadas`).
- **Checks:** every EU feature except the merged one matches the workbook one to one; `EU_LAU_CODE`
  equals `GISCO_ID`; every `NUTS3_ID` / `NUTS2_ID` exists in the NUTS subset (§1a); codes stay text
  (HU `02103` keeps its zero).
- `CITY ID` / `FUA ID` are not carried (not used by the pipeline). GDP needs nothing here: the NUTS
  subset files already hold it (§3), joined through `NUTS2_ID` / `NUTS3_ID`.
- Skips if the columns exist (`REBUILD_LAU_IDS = True` to recompute); must be re-run after any
  rebuild of §2a–2c.

In [11]:
from traintran.eu_lau_catchment import load_eu_lau

REBUILD_LAU_IDS = False            # True -> recompute even if the subset already has the columns
LAU_ID_XLSX = Path("data/eu/ref-nuts-2024-01m/EU-27-LAU-2025-NUTS-2024.xlsx")
LAU_ID_COLS = ["LAU_CODE", "EU_LAU_CODE", "LAU_NAME_NATIONAL", "LAU_NAME_LATIN",
               "NUTS3_ID", "NUTS2_ID", "LAU_ID_SOURCE"]

lau_subset = gpd.read_file(LAU_SUBSET_FILE)
if set(LAU_ID_COLS) <= set(lau_subset.columns) and not REBUILD_LAU_IDS:
    print(f"  [exists]    LAU identifiers already in {LAU_SUBSET_FILE} -- skipping "
          "(set REBUILD_LAU_IDS = True to recompute)")
else:
    lau_subset = lau_subset.drop(columns=LAU_ID_COLS, errors="ignore")
    wb = load_eu_lau(LAU_ID_XLSX, countries=LAU_SUBSET_COUNTRIES)
    assert wb["euc_lau_code"].is_unique, "duplicate EU LAU CODE in the workbook"
    wb = wb.set_index("euc_lau_code")

    is_ua = lau_subset["CNTR_CODE"] == "UA"
    is_merged = lau_subset["TERRITORIAL_NOTE"].fillna("").str.startswith("merged:")
    is_eu = ~is_ua & ~is_merged
    ids = pd.DataFrame(index=lau_subset.index, columns=LAU_ID_COLS, dtype=object)

    # EU: one-to-one on GISCO_ID == EU LAU CODE.
    g = lau_subset.loc[is_eu, "GISCO_ID"]
    missing = sorted(set(g) - set(wb.index))
    assert not missing, f"{len(missing)} EU features not in the workbook: {missing[:5]}"
    for col, src_col in [("LAU_CODE", "lau_code"), ("LAU_NAME_NATIONAL", "lau_name_national"),
                         ("LAU_NAME_LATIN", "lau_name_latin"), ("NUTS3_ID", "nuts3")]:
        ids.loc[is_eu, col] = g.map(wb[src_col]).to_numpy()
    ids.loc[is_eu, "EU_LAU_CODE"] = g.to_numpy()
    ids.loc[is_eu, "LAU_ID_SOURCE"] = "LAU_2025_NUTS_2024_xlsx"

    # Merged features (dissolved in 2c): codes from the new id, NUTS3 from the parts.
    for idx in lau_subset.index[is_merged]:
        parts = lau_subset.at[idx, "TERRITORIAL_NOTE"].split(":", 1)[1].strip().split(";")
        assert set(parts) <= set(wb.index), parts
        n3 = {wb.at[p, "nuts3"] for p in parts}
        assert len(n3) == 1, f"merged parts {parts} span several NUTS3: {n3}"
        gid = lau_subset.at[idx, "GISCO_ID"]
        name = lau_subset.at[idx, "LAU_NAME"]
        ids.loc[idx] = [gid.split("_", 1)[1], gid, name, name, n3.pop(), None,
                        "merged_2026: " + ";".join(parts)]

    # UA hromadas: no workbook rows; names and oblast from HDX.
    hrom = gpd.read_file(UA_HROMADAS, layer="hromadas", ignore_geometry=True).set_index("adm3_pcode")
    u = lau_subset.loc[is_ua, "GISCO_ID"]
    assert u.isin(hrom.index).all(), "UA feature without a hromada record"
    ids.loc[is_ua, "LAU_CODE"] = u.to_numpy()
    ids.loc[is_ua, "LAU_NAME_NATIONAL"] = u.map(hrom["adm3_name1"]).to_numpy()
    ids.loc[is_ua, "LAU_NAME_LATIN"] = u.map(hrom["adm3_name"]).to_numpy()
    ids.loc[is_ua, "NUTS2_ID"] = u.map(hrom["adm1_pcode"]).to_numpy()
    ids.loc[is_ua, "LAU_ID_SOURCE"] = "HDX_ukr_admin3"

    ids.loc[~is_ua, "NUTS2_ID"] = ids.loc[~is_ua, "NUTS3_ID"].str[:4]

    # Checks.
    _n3 = set(gpd.read_file(SUBSET_DIR / "NUTS_RG_01M_2024_4326_LEVL_3.geojson", ignore_geometry=True)["NUTS_ID"])
    _n2 = set(gpd.read_file(SUBSET_DIR / "NUTS_RG_01M_2024_4326_LEVL_2.geojson", ignore_geometry=True)["NUTS_ID"])
    assert (ids.loc[~is_ua, "EU_LAU_CODE"] == lau_subset.loc[~is_ua, "GISCO_ID"]).all()
    assert ids.loc[~is_ua, "NUTS3_ID"].isin(_n3).all(), sorted(set(ids.loc[~is_ua, "NUTS3_ID"]) - _n3)[:5]
    assert ids["NUTS2_ID"].isin(_n2).all(), sorted(set(ids["NUTS2_ID"].dropna()) - _n2)[:5]
    for c in ["LAU_CODE", "LAU_NAME_NATIONAL", "LAU_NAME_LATIN", "NUTS2_ID", "LAU_ID_SOURCE"]:
        assert ids[c].notna().all(), f"{c}: {int(ids[c].isna().sum())} empty"
    assert not lau_subset.assign(c=ids["LAU_CODE"]).duplicated(["CNTR_CODE", "c"]).any(), "LAU_CODE repeats"
    assert (ids.loc[is_eu, "LAU_CODE"] == g.str.split("_", n=1).str[1]).all(), "LAU_CODE != GISCO_ID suffix"

    for c in LAU_ID_COLS:
        lau_subset[c] = ids[c].astype(object)
    lau_subset.to_file(LAU_SUBSET_FILE, driver="GeoJSON")
    print(f"  [updated]   {LAU_SUBSET_FILE}  ({len(lau_subset):,} features; {', '.join(LAU_ID_COLS)})")

_cov = lau_subset.groupby("CNTR_CODE").agg(
    features=("GISCO_ID", "size"),
    LAU_CODE=("LAU_CODE", lambda s: int(s.notna().sum())),
    NUTS3_ID=("NUTS3_ID", lambda s: int(s.notna().sum())),
    NUTS2_ID=("NUTS2_ID", lambda s: int(s.notna().sum())),
    NSO_LAU_ID=("NSO_LAU_ID", lambda s: int(s.notna().sum())),
    example=("LAU_CODE", "first"))
display(_cov)

  [exists]    LAU identifiers already in data\eu\ref-lau-2024-01m\subsetv4atua\LAU_RG_01M_2024_4326.geojson -- skipping (set REBUILD_LAU_IDS = True to recompute)


,features,LAU_CODE,NUTS3_ID,NUTS2_ID,NSO_LAU_ID,example
CNTR_CODE,,,,,,
AT,2092,2092,2092,2092,2092,10605
CZ,6258,6258,6258,6258,6258,532088
HU,3155,3155,3155,3155,3155,12706
PL,2477,2477,2477,2477,2477,1002301600202
SK,2927,2927,2927,2927,2927,502111
UA,1769,1769,0,1769,0,UA0102003


## 3. Regional GDP (Eurostat `nama_10r_3gdp`) on the NUTS subset

Attaches GDP to every file of the NUTS 2024 subset (section 1a), so the explore notebooks read it
with `traintran.enrichment.load_nuts_gdp(cfg)` instead of parsing the spreadsheet themselves.

- **EU source:** `data/eu/nama_10r_3gdp__custom_15402261_spreadsheet.xlsx` -- a *custom* Eurostat
  export of `nama_10r_3gdp` (GDP at current market prices by NUTS 3 region; extracted 15/07/2026,
  dataset updated 10/02/2026). There is no stable download link for a custom export, so the file
  stays a manual input. It is NUTS-2024-coded and covers every subset region at levels 0-3.
  Units: `MIO_EUR` (million euro) and `PPS_EU27_2020_HAB` (PPS per inhabitant), 2021-2024.
- **UA source:** `data/ukraine/14-valovii-regionalnii-produkt-u-faktichnikh-tsinakh-mln-grn.xlsx`
  (State Statistics Service, gross regional product, million UAH), **2021 only**, converted at
  32.34 UAH/EUR -- the same construction as `input_data_extension_ukraine.ipynb`. Level 0 (`UA`)
  and level 2 (oblasts); Crimea and Sevastopol are `NaN` in the source. No PPS figure exists.
- **`POP_2021` / `POP_2024`:** summed LAU population -- 2021 from
  `EU-27-LAU-2021-Census-NUTS-2021.xlsx`, 2024 from `cfg.lau_file` (the LAU 2025 workbook).
  For UA, `POP_2021` is the hromada population of **1 Jan 2022** (`POP_2021_SOURCE` says so).
- **`GDP_PER_POP_<year>`** = `MIO_EUR_<year>` x 1e6 / `POP_<year>` (euro per inhabitant).
- `GDP_SOURCE` marks the rows: `ESTAT_nama_10r_3gdp` or `UkrStat_GRP_2021`.

In [12]:
import numpy as np
import pyogrio
from traintran.eu_lau_catchment import load_eu_lau

REBUILD_NUTS_GDP = False            # True -> recompute even if the subset already has GDP
NAMA_XLSX = Path("data/eu/nama_10r_3gdp__custom_15402261_spreadsheet.xlsx")
UA_GDP_XLSX = Path("data/ukraine/14-valovii-regionalnii-produkt-u-faktichnikh-tsinakh-mln-grn.xlsx")
LAU_2021_XLSX = Path("data/eu/EU-27-LAU-2021-Census-NUTS-2021.xlsx")
EXCHANGE_RATE_UAH_PER_EUR = 32.34
GDP_YEARS = [2021, 2022, 2023, 2024]
GDP_VALUE_COLS = ([f"MIO_EUR_{y}" for y in GDP_YEARS]
                  + [f"PPS_EU27_2020_HAB_{y}" for y in GDP_YEARS])
GDP_ADDED_COLS = GDP_VALUE_COLS + ["POP_2021", "POP_2024", "POP_2021_SOURCE",
                                   "GDP_PER_POP_2021", "GDP_PER_POP_2024", "GDP_SOURCE"]
NUTS_SUBSET_FILES = [SUBSET_DIR / f for f in NUTS_KEEP[:-1]]          # the 5 subset GeoJSONs

_fields = pyogrio.read_info(SUBSET_DIR / "NUTS_RG_01M_2024_4326_LEVL_0.geojson")["fields"]
NUTS_GDP_DONE = "MIO_EUR_2024" in list(_fields) and not REBUILD_NUTS_GDP

if NUTS_GDP_DONE:
    print(f"  [exists]    GDP already in {SUBSET_DIR} -- skipping (set REBUILD_NUTS_GDP = True to recompute)")
else:
    # --- EU: same header parsing the explore notebooks used. Row 7 = unit codes, merged across
    # each unit's 4-year block (ffill); row 9 = years; data from row 11; ':' = not available.
    raw = pd.read_excel(NAMA_XLSX, sheet_name="Data", header=None)
    unit_row = raw.iloc[6, 2:].ffill()
    year_row = raw.iloc[8, 2:]
    value_cols = [f"{unit}_{int(year)}" for unit, year in zip(unit_row, year_row)]
    assert value_cols == GDP_VALUE_COLS, value_cols
    gdp_eu = raw.iloc[10:].copy()
    gdp_eu.columns = ["geo_code", "geo_label"] + value_cols
    gdp_eu = gdp_eu.dropna(subset=["geo_code"])
    gdp_eu = gdp_eu[~gdp_eu["geo_code"].isin(["Special value", ":"])]
    gdp_eu = gdp_eu[gdp_eu["geo_code"].str[:2].isin(LAU_SUBSET_COUNTRIES)].reset_index(drop=True)
    gdp_eu[value_cols] = gdp_eu[value_cols].apply(pd.to_numeric, errors="coerce")
    gdp_eu["GDP_SOURCE"] = "ESTAT_nama_10r_3gdp"

    # --- UA: gross regional product 2021, million UAH -> million EUR. `code` must stay a string
    # ('0000000000' is the country total); row 2 repeats the header in Ukrainian.
    ua_raw = pd.read_excel(UA_GDP_XLSX, sheet_name="Валовий регіональний продукт",
                           header=0, skiprows=[1], dtype={"code": str})
    ua_raw = ua_raw[ua_raw["period"] == 2021].copy()
    gdp_ua = pd.DataFrame({
        "geo_code": ["UA" if c == "0000000000" else "UA" + c[:2] for c in ua_raw["code"]],
        "MIO_EUR_2021": pd.to_numeric(ua_raw["data"], errors="coerce").values / EXCHANGE_RATE_UAH_PER_EUR,
        "GDP_SOURCE": "UkrStat_GRP_2021",
    })
    for c in GDP_VALUE_COLS:
        if c not in gdp_ua:
            gdp_ua[c] = np.nan

    gdp = pd.concat([gdp_eu.drop(columns="geo_label"), gdp_ua], ignore_index=True)
    assert gdp["geo_code"].is_unique

    # --- Population at every NUTS level (nuts3 truncated to 2/3/4/5 chars).
    def _pop_by_nuts(lau_xlsx):
        lau = load_eu_lau(lau_xlsx, countries=LAU_SUBSET_COUNTRIES)
        return pd.concat([lau.groupby(lau["nuts3"].str[:n])["population"].sum() for n in (2, 3, 4, 5)])

    hrom = gpd.read_file(UA_HROMADAS, layer="hromadas", ignore_geometry=True)
    ua_pop = hrom.groupby("adm1_pcode")["population"].sum(min_count=1)
    ua_pop["UA"] = hrom["population"].sum()
    is_ua = gdp["geo_code"].str.startswith("UA")
    gdp["POP_2021"] = gdp["geo_code"].map(_pop_by_nuts(LAU_2021_XLSX))
    gdp.loc[is_ua, "POP_2021"] = gdp.loc[is_ua, "geo_code"].map(ua_pop)
    gdp["POP_2024"] = gdp["geo_code"].map(_pop_by_nuts(cfg.lau_file))
    gdp["POP_2021_SOURCE"] = np.where(is_ua, "UkrStat_hromadas_2022-01-01", "EU_LAU_2021_census")
    for y in (2021, 2024):
        gdp[f"GDP_PER_POP_{y}"] = gdp[f"MIO_EUR_{y}"] * 1_000_000 / gdp[f"POP_{y}"]

    print(f"  GDP table: {len(gdp):,} regions ({int((~is_ua).sum())} EU, {int(is_ua.sum())} UA)")

C:\Users\csb\anaconda\envs\traintranv2\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  GDP table: 250 regions (222 EU, 28 UA)


In [13]:
if not NUTS_GDP_DONE:
    for path in NUTS_SUBSET_FILES:
        gdf = gpd.read_file(path).drop(columns=GDP_ADDED_COLS, errors="ignore")
        gdf = gdf.merge(gdp[["geo_code"] + GDP_ADDED_COLS], left_on="NUTS_ID",
                        right_on="geo_code", how="left").drop(columns="geo_code")
        gdf.to_file(path, driver="GeoJSON")
        print(f"  [updated]   {path}")

# Coverage per level x country: regions, with GDP, with GDP per head (any year).
_rows = []
for n in range(4):
    g = load_nuts_level(cfg, n)
    _has = g["mio_eur_2024"].notna() | g["mio_eur_2021"].notna()
    _pc = g["gdp_per_pop_2024"].notna() | g["gdp_per_pop_2021"].notna()
    _rows.append(pd.DataFrame({"level": n, "country": g["cntr_code"], "gdp": _has, "per_head": _pc}))
(pd.concat(_rows).groupby(["level", "country"])
   .agg(regions=("gdp", "size"), with_gdp=("gdp", "sum"), with_gdp_per_head=("per_head", "sum")))

  [updated]   data\eu\ref-nuts-2024-01m\subsetv4atua\NUTS_RG_01M_2024_4326.geojson
  [updated]   data\eu\ref-nuts-2024-01m\subsetv4atua\NUTS_RG_01M_2024_4326_LEVL_0.geojson


  [updated]   data\eu\ref-nuts-2024-01m\subsetv4atua\NUTS_RG_01M_2024_4326_LEVL_1.geojson


  [updated]   data\eu\ref-nuts-2024-01m\subsetv4atua\NUTS_RG_01M_2024_4326_LEVL_2.geojson


  [updated]   data\eu\ref-nuts-2024-01m\subsetv4atua\NUTS_RG_01M_2024_4326_LEVL_3.geojson


regions  with_gdp  with_gdp_per_head
level country                                      
0     AT             1         1                  1
      CZ             1         1                  1
      HU             1         1                  1
      PL             1         1                  1
      SK             1         1                  1
      UA             1         1                  1
1     AT             3         3                  3
      CZ             1         1                  1
      HU             3         3                  3
      PL             7         7                  7
      SK             1         1                  1
2     AT             9         9                  9
      CZ             8         8                  8
      HU             8         8                  8
      PL            17        17                 17
      SK             4         4                  4
      UA            27        25                 25
3     AT            35        35                 35
      CZ            14        14                 14
      HU            20        20                 20
      PL            73        73                 73
      SK             8         8                  8